In [37]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [39]:
df = pd.read_csv(r"PhiUSIIL_Phishing_URL_Dataset.csv")
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [40]:
cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']

df.drop(columns=cols_to_drop, inplace=True)


In [ ]:
cols_to_drop = ['NoOfLettersInURL', 'DomainTitleMatchScore', 'URLSimilarityIndex']

df.drop(columns=cols_to_drop, inplace=True)

In [58]:
corr = df.corr()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 2)))

high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
for pair in high_corr:
    print(pair)

('URLLength', 'NoOfDegitsInURL', np.float64(0.84))
('NoOfDegitsInURL', 'NoOfEqualsInURL', np.float64(0.81))
('HasObfuscation', 'ObfuscationRatio', np.float64(0.8))
('NoOfObfuscatedChar', 'NoOfAmpersandInURL', np.float64(0.79))
('URLLength', 'NoOfOtherSpecialCharsInURL', np.float64(0.78))
('NoOfEqualsInURL', 'NoOfOtherSpecialCharsInURL', np.float64(0.78))
('NoOfDegitsInURL', 'NoOfOtherSpecialCharsInURL', np.float64(0.77))
('NoOfObfuscatedChar', 'NoOfEqualsInURL', np.float64(0.75))
('NoOfObfuscatedChar', 'NoOfDegitsInURL', np.float64(0.72))
('CharContinuationRate', 'SpacialCharRatioInURL', np.float64(-0.71))
('URLCharProb', 'DegitRatioInURL', np.float64(-0.71))
('NoOfSelfRef', 'NoOfExternalRef', np.float64(0.7))


In [60]:
corr_with_label = df.corr()['label'].abs().sort_values(ascending=False)
print(corr_with_label)


label                         1.000000
HasDescription                0.690232
IsHTTPS                       0.609132
HasSubmitButton               0.578561
IsResponsive                  0.548608
URLTitleMatchScore            0.539419
SpacialCharRatioInURL         0.533537
HasHiddenFields               0.507731
HasFavicon                    0.493711
URLCharProb                   0.469749
CharContinuationRate          0.467735
HasTitle                      0.459725
DegitRatioInURL               0.432032
Robots                        0.392620
NoOfJS                        0.373500
LetterRatioInURL              0.367794
Pay                           0.359747
NoOfOtherSpecialCharsInURL    0.358891
NoOfSelfRef                   0.316211
DomainLength                  0.283152
NoOfImage                     0.274658
LineOfCode                    0.272257
NoOfExternalRef               0.258627
URLLength                     0.233445
NoOfiFrame                    0.225822
Bank                     

In [21]:
binary_cols = ['IsDomainIP', 'HasObfuscation', 'IsHTTPS', 'HasExternalFormSubmit',
               'HasPasswordField', 'HasHiddenFields', 'Bank', 'Pay', 'Crypto']

for col in binary_cols:
    rates = df.groupby(col)['label'].mean()
    print(f"{col}:\n{rates}\n")

IsDomainIP:
IsDomainIP
0    0.573447
1    0.000000
Name: label, dtype: float64

HasObfuscation:
HasObfuscation
0    0.573074
1    0.000000
Name: label, dtype: float64

IsHTTPS:
IsHTTPS
0    0.00000
1    0.73074
Name: label, dtype: float64

HasExternalFormSubmit:
HasExternalFormSubmit
0    0.554109
1    0.958446
Name: label, dtype: float64

HasPasswordField:
HasPasswordField
0    0.548819
1    0.774478
Name: label, dtype: float64

HasHiddenFields:
HasHiddenFields
0    0.376131
1    0.894301
Name: label, dtype: float64

Bank:
Bank
0    0.536220
1    0.816932
Name: label, dtype: float64

Pay:
Pay
0    0.472686
1    0.891277
Name: label, dtype: float64

Crypto:
Crypto
0    0.564253
1    0.889792
Name: label, dtype: float64



In [54]:
X = df.drop('label', axis=1).values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [55]:
class PhishingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = PhishingDataset(X_train, y_train)
test_dataset = PhishingDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

class DeepNN(nn.Module):
    def __init__(self, input_dim):
        super(DeepNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            # nn.BatchNorm1d(256),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(32, 64),
            # nn.BatchNorm1d(128),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(64, 32),
            # nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepNN(input_dim=X_train.shape[1]).to(device)


In [56]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/20 - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/20 - Loss: 0.0115
Epoch 2/20 - Loss: 0.0024
Epoch 3/20 - Loss: 0.0019
Epoch 4/20 - Loss: 0.0015
Epoch 5/20 - Loss: 0.0014
Epoch 6/20 - Loss: 0.0012
Epoch 7/20 - Loss: 0.0010
Epoch 8/20 - Loss: 0.0009
Epoch 9/20 - Loss: 0.0009
Epoch 10/20 - Loss: 0.0009
Epoch 11/20 - Loss: 0.0009
Epoch 12/20 - Loss: 0.0007
Epoch 13/20 - Loss: 0.0008
Epoch 14/20 - Loss: 0.0006
Epoch 15/20 - Loss: 0.0007
Epoch 16/20 - Loss: 0.0006
Epoch 17/20 - Loss: 0.0006
Epoch 18/20 - Loss: 0.0006
Epoch 19/20 - Loss: 0.0006
Epoch 20/20 - Loss: 0.0005


In [57]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = torch.argmax(model(X_batch), dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20124
           1       1.00      1.00      1.00     27035

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

[[20121     3]
 [    3 27032]]
